# Gene Coordinate Annotation


## Description

We use a gene coordinate annotation pipeline based on [`pyqtl`, as demonstrated here](https://github.com/broadinstitute/gtex-pipeline/blob/master/qtl/src/eqtl_prepare_expression.py). This adds genomic coordinate annotations to gene-level molecular phenotype files generated in `gct` format and converts them to `bed` format for downstreams analysis.


### Alternative implementation

Previously we use `biomaRt` package in R instead of code from `pyqtl`. The core function calls are:

```r
    ensembl = useEnsembl(biomart = "ensembl", dataset = "hsapiens_gene_ensembl", version = "$[ensembl_version]")
    ensembl_df <- getBM(attributes=c("ensembl_gene_id","chromosome_name", "start_position", "end_position"),mart=ensembl)
```

We require ENSEMBL version to be specified explicitly in this pipeline. As of 2021 for the Brain xQTL project, we use ENSEMBL version 103.

## Input

1. Molecular phenotype data in `gct` format, with the first column being ENSEMBL ID and other columns being sample names. 
2. GTF for collapsed gene model
    - the gene names must be consistent with the molecular phenotype data matrices (eg ENSG00000000003 vs. ENSG00000000003.1 will not work) 
3. (Optional) Meta-data to match between sample names in expression data and genotype files
    - Tab delimited with header
    - Only 2 columns: first column is sample name in expression data, 2nd column is sample name in genotype data
    - **must contains all the sample name in expression matrices even if they don't existing in genotype data**
    

## Output

Molecular phenotype data in `bed` format.

## Minimal Working Example Steps

These commands assume the MWE data bundle is available as `mwe_data/` in the repository root. Run each command from the repository root; commands that reference `output/mwe_notebook/` expect the upstream MWE commands in this section to have produced those files.


### Coordinate annotation for a gene-expression BED

Use the staged synthetic expression BED and the MWE-compatible GTF.


In [ ]:
sos run pipeline/gene_annotation.ipynb annotate_coord \
    --cwd output/mwe_notebook/phenotype/annotation \
    --phenoFile mwe_data/xqtl_hg_synthetic.expression.bed.gz \
    --coordinate-annotation mwe_data/rnaseqc_compatible.gtf \
    --phenotype-id-column gene_id \
    --modular-script-dir code/script


## Troubleshooting

| Step | Substep | Problem | Possible Reason | Solution |
|------|---------|---------|------------------|---------|
|  |  |  |  |  |




## Command Interface

In [2]:
!sos run gene_annotation.ipynb -h

usage: sos run gene_annotation.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  targets:              One or more targets to generate
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  region_list_generation
  annotate_coord
  annotate_coord_biomart
  map_leafcutter_cluster_to_gene
  annotate_leafcutter_isoforms
  annotate_psichomics_isoforms

Global Workflow Options:
  --cwd output (as path)
                        Work directory & output directory
  --phenoFile VAL (as path, required)
                        Molecular phenotype matrix
  --job-size 1 (as int)
                        For cluster jobs, number commands to run per job
  --walltime 5h
                        Wall clock time expected
  --mem 16G
                        Memory expected
  --numThreads 1 (as 

## Setup and global parameters

In [ ]:
[global]
parameter: modular_script_dir = path('code/script')  # override with --modular-script-dir
# Work directory & output directory
parameter: cwd = path("output")
# Molecular phenotype matrix
parameter: phenoFile = path
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 1
parameter: container = ""
parameter: entrypoint= ""

## Annotate coord

### Input:

1. A gtf file used to generated the bed
2. A phenotype bed file, must have a gene_id column indicating the name of genes.

### Output:

1. annotated phenotype bed file
2. Optional: a tsv file only has the first several columns indicating phenotype_id and names. For RNA-seq, it contains five columns: chr,start,end,gene_id,gene_name

## Implementation using `pyqtl`

Implementation based on [GTEx pipeline](https://github.com/broadinstitute/gtex-pipeline/blob/master/qtl/src/eqtl_prepare_expression.py).

Following step serves to annotate coord for gene expression file.

In [ ]:
[annotate_coord]
# A file to map sample ID from expression to genotype, must contain two columns, sample_id and participant_id, mapping IDs in the expression files to IDs in the genotype (these can be the same).
parameter: sample_participant_lookup = path()
# Options: gene, protein, atac
parameter: molecular_trait_type = "gene" 
# gtf annotation for RNA-seq data; other types of annotation index for protein and atac-seq
parameter: coordinate_annotation = path
# gene_id or gene_name for RNA-seq data, SOMAseqID for proteins for example
parameter: phenotype_id_column = "gene_id"
# Optional mapping file (protein_name_index for protein, etc.)
parameter: auxiliary_id_mapping = path()
# Optional processing flag
parameter: strip_id = False
parameter: sep = "\t"

input: phenoFile, coordinate_annotation
output: f'{cwd:a}/{_input[0]:bn}.bed.gz', f'{cwd:a}/{_input[0]:bn}.region_list.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, tags = f'{step_name}_{_output[0]:bn}'
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container, entrypoint = entrypoint
    python3 ${modular_script_dir}/data_preprocessing/phenotype/gene_annotation.py \
        --step annotate_coord \
        --phenoFile "${_input[0]}" \
        --coordinate-annotation "${_input[1]}" \
        --sample-participant-lookup "${sample_participant_lookup}" \
        --molecular-trait-type "${molecular_trait_type}" \
        --phenotype-id-column "${phenotype_id_column}" \
        --auxiliary-id-mapping "${auxiliary_id_mapping}" \
        --sep "${sep}" \
        ${"--strip-id" if strip_id else ""} \
        --output-bed "${_output[0]}" \
        --output-region-list "${_output[1]}" \
        --gene-list-output "${cwd:a}/${_input[0]:bn}.gene_list.tsv"


## Implementation using biomaRt
This workflow adds the annotations of chr pos(TSS where start = end -1) and gene_ID to the `bed` file. **This workflow is obsolete**.

In [ ]:
[annotate_coord_biomart]
parameter: ensembl_version=int
input: phenoFile
output: f'{cwd:a}/{_input:bn}.bed.gz',
        f'{cwd:a}/{_input:bn}.region_list.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container, entrypoint = entrypoint
    Rscript ${modular_script_dir}/data_preprocessing/phenotype/gene_annotation.R \
        --step annotate_coord_biomart \
        --phenoFile "${_input[0]}" \
        --ensembl-version ${ensembl_version} \
        --output-bed "${_output[0]}" \
        --output-region-list "${_output[1]}"


## Annotation of leafcutter isoform
The following steps processed the output files of leafcutter so that they are TensorQTL ready. Shown below are three intemediate files

Exon list

chr   |  start  |  end    |  strand  | gene_id | gene_name
------|---------|---------|----------|----------|--------------
chr1  |  29554  |  30039  |  +       | ENSG00000243485 | MIR1302-2HG
chr1  |  30564  |  30667  |  +       | ENSG00000243485 | MIR1302-2HG
chr1  |  30976  |  31097  |  +       | ENSG00000243485 | MIR1302-2HG
chr1  |  35721  |  36081  |  -       | ENSG00000237613 | FAM138A
chr1  |  35277  |  35481  |  -       | ENSG00000237613 | FAM138A
chr1  |  34554  |  35174  |  -       | ENSG00000237613 | FAM138A
chr1  |  65419  |  65433  |  +       | ENSG00000186092 | OR4F5
chr1  |  65520  |  65573  |  +       | ENSG00000186092 | OR4F5
chr1  |  69037  |  71585  |  +       | ENSG00000186092 | OR4F5

clusters_to_genes


|clu    | genes |
|--------|---------- |
|1:clu_1_+  |    ENSG00000116288|
|1:clu_10_+ |    ENSG00000143774|
|1:clu_11_+ |    ENSG00000143774|
|1:clu_12_+ |    ENSG00000143774|
|1:clu_14_- |    ENSG00000126709|
|1:clu_15_- |    ENSG00000121753|
|1:clu_16_- |    ENSG00000121753|
|1:clu_17_- |    ENSG00000116560|
|1:clu_18_- |    ENSG00000143549|

phenotype_group

|X1|X2|
|-------------|---|
| 7:102476270:102478811:clu_309_-:ENSG00000005075 | ENSG00000005075 | 
| 7:102476270:102478808:clu_309_-:ENSG00000005075 | ENSG00000005075 |
| X:47572961:47574002:clu_349_-:ENSG00000008056   | ENSG00000008056 |
| X:47572999:47574002:clu_349_-:ENSG00000008056   | ENSG00000008056 |
| 8:27236905:27239971:clu_322_-:ENSG00000015592   | ENSG00000015592 |
| 8:27239279:27239971:clu_322_-:ENSG00000015592   | ENSG00000015592 |
| 8:27241262:27241677:clu_323_-:ENSG00000015592   | ENSG00000015592 |
| 8:27241262:27242397:clu_323_-:ENSG00000015592   | ENSG00000015592 |
| 8:27241757:27242397:clu_323_-:ENSG00000015592   | ENSG00000015592 |
| 1:35558223:35559107:clu_4_+:ENSG00000020129     | ENSG00000020129 |

The gtf used here should be the collapsed gtf, i.e. the final output of reference_data gtf processing and the one used to called rnaseq.

In [ ]:
[map_leafcutter_cluster_to_gene]
## Extract the code in case psichromatic needs to be processed the same way
## PheoFile in this step is the intron_count file
parameter: intron_count = path
# Defines the mapping strategy options: 'site' or 'region', with 'site' as the default. 
# The 'site' strategy maps introns to the start and end of each exon. 
# The 'region' strategy, to be recommended in leafcutter2, maps each intron based on it overlapping more than overlap_ratio of a gene's region.
parameter: map_stra = "site" 
# Define the overlap ratio as the proportion of the cluster length that intersects with a gene, used to determine mapping to the gene.
parameter: overlap_ratio = 0.8
input: intron_count, annotation_gtf
output: f'{cwd}/{_input[0]:b}.exon_list', f'{cwd}/{_input[0]:b}.leafcutter.clusters_to_genes.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container, entrypoint = entrypoint
    python3 ${modular_script_dir}/data_preprocessing/phenotype/gene_annotation.py \
        --step map_leafcutter_cluster_to_gene \
        --intron-count "${_input[0]}" \
        --annotation-gtf "${_input[1]}" \
        --map-stra "${map_stra}" \
        --overlap-ratio ${overlap_ratio} \
        --output-exon-list "${_output[0]}" \
        --output-cluster-map "${_output[1]}"


In [ ]:
[annotate_leafcutter_isoforms]
parameter: sample_participant_lookup = path()
input: phenoFile, annotation_gtf,output_from("map_leafcutter_cluster_to_gene")
output: f'{cwd:a}/{_input[0]:bn}.formated.bed.gz', f'{cwd:a}/{_input[0]:bn}.phenotype_group.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container, entrypoint = entrypoint
    python3 ${modular_script_dir}/data_preprocessing/phenotype/gene_annotation.py \
        --step annotate_leafcutter_isoforms \
        --phenoFile "${_input[0]}" \
        --annotation-gtf "${_input[1]}" \
        --cluster-map "${_input[3]}" \
        --sample-participant-lookup "${sample_participant_lookup}" \
        --output-bed "${_output[0]}" \
        --output-phenotype-group "${_output[1]}"


## Processing of psichomics output
It occurs that the psichomatic by default grouped the isoforms by gene name, so only thing needs to be done is to extract this information and potentially renamed the gene symbol into ENSG ID

In [ ]:
[annotate_psichomics_isoforms]
parameter: sample_participant_lookup = path()
input: phenoFile, annotation_gtf
output: f'{cwd:a}/{_input[0]:bn}.formated.bed.gz', f'{cwd:a}/{_input[0]:bn}.phenotype_group.txt'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output[0]:bn}'  
bash: expand= "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout', container = container, entrypoint = entrypoint
    python3 ${modular_script_dir}/data_preprocessing/phenotype/gene_annotation.py \
        --step annotate_psichomics_isoforms \
        --phenoFile "${_input[0]}" \
        --annotation-gtf "${_input[1]}" \
        --sample-participant-lookup "${sample_participant_lookup}" \
        --output-bed "${_output[0]}" \
        --output-phenotype-group "${_output[1]}"
